# Exploratory Data Analysis

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import os
os.makedirs("../data/charts", exist_ok=True)

In [2]:
# Load data
df = pd.read_parquet("../data/accidents_features.parquet")
print(f"Shape: {df.shape}")

Shape: (500000, 59)


In [3]:
# Chart 1 — Accidents by Hour
hourly = df.groupby('hour_of_day').size().reset_index(name='count')

fig = px.bar(hourly, x='hour_of_day', y='count', 
             title="Accident Frequency by Hour of Day",
             color='count', color_continuous_scale='Reds')
fig.add_vline(x=7, line_dash="dash", line_color="blue", annotation_text="Peak")
fig.add_vline(x=9, line_dash="dash", line_color="blue")
fig.add_vline(x=17, line_dash="dash", line_color="blue", annotation_text="Peak")
fig.add_vline(x=20, line_dash="dash", line_color="blue")

fig.write_html("../data/charts/chart_hourly.html")
print("Saved: chart_hourly.html")

Saved: chart_hourly.html


In [4]:
# Chart 2 — Severity Distribution
severity_counts = df['Severity'].value_counts().sort_index()

fig = go.Figure(data=[go.Pie(
    labels=['Severity 1', 'Severity 2', 'Severity 3', 'Severity 4'],
    values=severity_counts.values,
    hole=0.4,
    marker=dict(colors=['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c'])
)])
fig.update_layout(title="Accident Severity Distribution")
fig.write_html("../data/charts/chart_severity.html")
print("Saved: chart_severity.html")

Saved: chart_severity.html


In [5]:
# Chart 3 — Accidents by Day of Week
day_map = {0: 'Mon', 1: 'Tue', 2: 'Wed', 3: 'Thu', 4: 'Fri', 5: 'Sat', 6: 'Sun'}
df['day_name'] = df['day_of_week'].map(day_map)
day_order = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

dow = df['day_of_week'].value_counts().reindex(range(7), fill_value=0)
dow.index = day_order

colors = ['#3498db'] * 5 + ['#e74c3c'] * 2  # Highlight Sat/Sun
fig = px.bar(x=day_order, y=dow.values, color=day_order,
             title="Accidents by Day of Week",
             labels={'x': 'Day', 'y': 'Count'})
fig.update_layout(showlegend=False)
fig.write_html("../data/charts/chart_weekday.html")
print("Saved: chart_weekday.html")

Saved: chart_weekday.html


In [6]:
# Chart 4 — Weather Risk vs Accident Count
weather = df.groupby('weather_risk_score').agg(
    count=('Severity', 'count'),
    avg_severity=('Severity', 'mean')
).reset_index()

fig = px.bar(weather, x='weather_risk_score', y='count',
             title="Accident Count by Weather Risk Score",
             labels={'weather_risk_score': 'Weather Risk (1-5)', 'count': 'Accident Count'})
fig.write_html("../data/charts/chart_weather.html")
print("Saved: chart_weather.html")

Saved: chart_weather.html


In [7]:
# Chart 5 — Monthly Trend
monthly = df.groupby(['year', 'month']).size().reset_index(name='count')
monthly['year_month'] = monthly['year'].astype(str) + '-' + monthly['month'].astype(str).str.zfill(2)
monthly = monthly.sort_values('year_month')

fig = px.line(monthly, x='year_month', y='count',
              title="Monthly Accident Trend",
              labels={'year_month': 'Month', 'count': 'Accidents'})
fig.update_traces(mode='lines+markers')
fig.write_html("../data/charts/chart_trend.html")
print("Saved: chart_trend.html")

Saved: chart_trend.html


In [8]:
# Chart 6 — Hour vs Day Heatmap
pivot = df.pivot_table(index='hour_of_day', columns='day_of_week', 
                        aggfunc='size', fill_value=0)

fig = go.Figure(data=go.Heatmap(
    z=pivot.values,
    x=['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'],
    y=pivot.index,
    colorscale='Reds'
))
fig.update_layout(title="Accidents by Hour and Day of Week",
                  xaxis_title="Day of Week", yaxis_title="Hour of Day")
fig.write_html("../data/charts/chart_heatmap.html")
print("Saved: chart_heatmap.html")

Saved: chart_heatmap.html


In [9]:
# Chart 7 — Top 15 States
if 'State' in df.columns:
    states = df['State'].value_counts().head(15).reset_index()
    states.columns = ['State', 'count']
    
    fig = px.bar(states, y='State', x='count', orientation='h',
                 title="Top 15 States by Accident Count",
                 labels={'State': 'State', 'count': 'Accidents'})
    fig.update_layout(yaxis={'categoryorder': 'total ascending'})
    fig.write_html("../data/charts/chart_states.html")
    print("Saved: chart_states.html")
else:
    print("State column not found, skipping chart_states.html")

Saved: chart_states.html


In [10]:
# Save summary stats as JSON for dashboard KPI cards
import json

hourly = df.groupby('hour_of_day').size()
most_dangerous_hour = int(hourly.idxmax())

weather_severity = df.groupby('weather_risk_score')['Severity'].mean()
highest_risk_weather = int(weather_severity.idxmax())

high_risk_pct = df['binary_severity'].mean() * 100

stats = {
    "total_records": int(len(df)),
    "most_dangerous_hour": most_dangerous_hour,
    "highest_risk_weather": highest_risk_weather,
    "high_risk_pct": f"{high_risk_pct:.1f}%"
}

with open("../data/charts/stats.json", 'w') as f:
    json.dump(stats, f, indent=2)

print("Saved: stats.json")
print(stats)

Saved: stats.json
{'total_records': 500000, 'most_dangerous_hour': 8, 'highest_risk_weather': 5, 'high_risk_pct': '37.2%'}


## Top 5 Insights

1. **Peak Hours**: Accidents peak during rush hours (7-9 AM and 5-7 PM), with the highest volume around 8 AM and 5 PM, indicating commuter-related incidents.

2. **Weekend vs Weekday**: Weekdays show higher accident counts than weekends, with Friday being the most accident-prone day.

3. **Severity Distribution**: Most accidents are Severity 2 or 3, indicating moderate impact. Severe accidents (Severity 4) represent a smaller but significant portion.

4. **Weather Impact**: Higher weather risk scores correlate with more accidents, but the relationship isn't linear — moderate risk (3) shows the highest count.

5. **Geographic Concentration**: California (CA), Texas (TX), and Florida (FL) dominate accident counts, reflecting population density and road network size.